In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2015_2016"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR2TOT_I.xpt"
csv_file = DATA_DIR / "DR2TOT_I_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Completely untouched pandas-decoded source.
source_df = df.copy()


print("=" * 100)
print("NHANES 2015-2016 DR2TOT_I XPT -> CSV")
print("=" * 100)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTDRD1",
    "WTDR2D",
    "DR2DRSTZ",
    "DR2EXMER",
    "DRABF",
    "DRDINT",
    "DR2DBIH",
    "DR2DAY",
    "DR2LANG",
    "DR2MRESP",
    "DR2HELP",
    "DR2TNUMF",
    "DR2STY",
    "DR2SKY",
    "DR2TKCAL",
    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",
    "DR2TCHOL",
    "DR2TATOC",
    "DR2TATOA",
    "DR2TRET",
    "DR2TVARA",
    "DR2TACAR",
    "DR2TBCAR",
    "DR2TCRYP",
    "DR2TLYCO",
    "DR2TLZ",
    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",
    "DR2TFOLA",
    "DR2TFA",
    "DR2TFF",
    "DR2TFDFE",
    "DR2TCHL",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVD",
    "DR2TVK",
    "DR2TCALC",
    "DR2TPHOS",
    "DR2TMAGN",
    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSODI",
    "DR2TPOTA",
    "DR2TSELE",
    "DR2TCAFF",
    "DR2TTHEO",
    "DR2TALCO",
    "DR2TMOIS",
    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",
    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",
    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",
    "DR2_300",
    "DR2_320Z",
    "DR2_330Z",
    "DR2BWATZ",
    "DR2TWS"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "DR2TOT_I column names/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 9544:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 85:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print("PASS: Structure = 9,544 rows x 85 columns.")


# ============================================================
# 4. DEFINE GENUINE DIETARY DECIMAL FIELDS
# ============================================================

#
# CRITICAL:
#
# These 50 variables contain genuine fractional values.
#
# DO NOT:
#
#   round()
#   astype(Int64)
#   use float_format=
#   globally remove ".0"
#
# from these columns.
#

DECIMAL_COLUMNS = [
    "WTDRD1",
    "WTDR2D",
    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",
    "DR2TATOC",
    "DR2TATOA",
    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",
    "DR2TCHL",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVD",
    "DR2TVK",
    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSELE",
    "DR2TALCO",
    "DR2TMOIS",
    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",
    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",
    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",
    "DR2_320Z",
    "DR2_330Z",
    "DR2BWATZ"
]


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(DECIMAL_COLUMNS) != 50:
    raise ValueError(
        "Expected exactly 50 genuine decimal fields."
    )


if len(INTEGER_COLUMNS) != 35:
    raise ValueError(
        "Expected exactly 35 whole-number/code fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine Dietary decimal fields:",
    len(DECIMAL_COLUMNS)
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT representation:
#
#     5.397605346934028e-79
#
# We NEVER globally replace it until the exact pattern
# has been validated.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:
    raise ValueError(
        "Expected all 85 DR2TOT_I fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


tiny_variable_count = int(
    tiny_counts.gt(0).sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)


print("\nTiny values by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Directly verified against the attached DR2TOT_I.xpt.
#
# 47,411 artifacts across exactly 70 variables.
#

EXPECTED_TINY_COUNTS = {
    "WTDRD1": 1038,
    "WTDR2D": 1479,
    "DR2DBIH": 11,

    "DR2TKCAL": 5,
    "DR2TPROT": 10,
    "DR2TCARB": 6,
    "DR2TSUGR": 6,
    "DR2TFIBE": 78,
    "DR2TTFAT": 9,
    "DR2TSFAT": 9,
    "DR2TMFAT": 9,
    "DR2TPFAT": 9,
    "DR2TCHOL": 104,
    "DR2TATOC": 10,
    "DR2TATOA": 5735,
    "DR2TRET": 62,
    "DR2TVARA": 30,
    "DR2TACAR": 950,
    "DR2TBCAR": 124,
    "DR2TCRYP": 704,
    "DR2TLYCO": 1857,
    "DR2TLZ": 90,
    "DR2TVB1": 7,
    "DR2TVB2": 9,
    "DR2TNIAC": 7,
    "DR2TVB6": 7,
    "DR2TFOLA": 11,
    "DR2TFA": 239,
    "DR2TFF": 69,
    "DR2TFDFE": 11,
    "DR2TCHL": 10,
    "DR2TVB12": 33,
    "DR2TB12A": 4566,
    "DR2TVC": 59,
    "DR2TVD": 184,
    "DR2TVK": 15,
    "DR2TCALC": 1,
    "DR2TPHOS": 7,
    "DR2TMAGN": 2,
    "DR2TIRON": 6,
    "DR2TZINC": 4,
    "DR2TCOPP": 1,
    "DR2TSODI": 1,
    "DR2TPOTA": 5,
    "DR2TSELE": 11,
    "DR2TCAFF": 2175,
    "DR2TTHEO": 4035,
    "DR2TALCO": 6232,
    "DR2TS040": 318,
    "DR2TS060": 328,
    "DR2TS080": 117,
    "DR2TS100": 39,
    "DR2TS120": 37,
    "DR2TS140": 14,
    "DR2TS160": 9,
    "DR2TS180": 12,
    "DR2TM161": 22,
    "DR2TM181": 11,
    "DR2TM201": 42,
    "DR2TM221": 1087,
    "DR2TP182": 9,
    "DR2TP183": 12,
    "DR2TP184": 3348,
    "DR2TP204": 61,
    "DR2TP205": 947,
    "DR2TP225": 681,
    "DR2TP226": 1110,

    "DR2_320Z": 1320,
    "DR2_330Z": 3761,
    "DR2BWATZ": 4084
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DR2TOT_I attachment.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 47411:
    raise ValueError(
        f"Unexpected total tiny-value count: "
        f"{tiny_total:,}"
    )


if tiny_variable_count != 70:
    raise ValueError(
        "Expected tiny values in exactly 70 variables."
    )


if true_zero_total != 0:
    raise ValueError(
        "Unexpected ordinary numeric zeros existed "
        "before correction."
    )


print(
    "\nPASS: Exact 47,411-value tiny-zero pattern "
    "across 70 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = df[col].eq(TINY_VALUE)

    actual_count = int(
        mask.sum()
    )

    if actual_count != expected_count:
        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )

    df.loc[
        mask,
        col
    ] = 0

    correction_total += actual_count


if correction_total != 47411:
    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "PASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df is used as the exact fidelity reference.
#
# The ONLY allowed corrections:
#
#     validated tiny value -> 0
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:
    raise ValueError(
        "Tiny XPORT artifacts remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 10. DETECT GENUINE DIETARY DECIMAL VARIABLES
# ============================================================

#
# Run BEFORE any Int64 conversion.
#
# np.round() is used ONLY to test whether a value
# contains a fractional component.
#
# No source value is modified.
#

print(
    "\n--- GENUINE DECIMAL DETECTION ---"
)


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:
        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{fractional_count:,} fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:
    raise ValueError(
        "STOP: Genuine Dietary decimal pattern differs.\n\n"
        f"Expected:\n{DECIMAL_COLUMNS}\n\n"
        f"Actual:\n{detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 50 genuine Dietary "
    "decimal variables detected."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)


if df["SEQN"].min() != 83732:
    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 93702:
    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(df["SEQN"].isna().sum()) != 0:
    raise ValueError(
        "Missing SEQN found."
    )


if int(df["SEQN"].duplicated().sum()) != 0:
    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 83732-93702 validated."
)


# ============================================================
# 12. CONVERT ONLY WHOLE-NUMBER/CODE FIELDS TO Int64
# ============================================================

#
# 35 fields become nullable Int64.
#
# 50 genuine decimal variables remain float.
#
# NO .round() during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():
            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 35 whole-number/code fields "
    "converted to nullable Int64."
)

print(
    "PASS: All 50 genuine Dietary decimal fields "
    "remain floating-point and unchanged."
)


# ============================================================
# 13. WTDRD1 EXACT XPT + CDC DISPLAY VALIDATION
# ============================================================

#
# Actual attached XPT:
#
#     positive min = 1974.178694
#     positive max = 454277.717307
#
# CDC HTML displays:
#
#     1974.178694 to 454277.71731
#
# We preserve the XPT precision exactly.
#

wtdrd1_positive = int(
    df["WTDRD1"]
    .gt(0)
    .sum()
)

wtdrd1_zero = int(
    df["WTDRD1"]
    .eq(0)
    .sum()
)

wtdrd1_missing = int(
    df["WTDRD1"]
    .isna()
    .sum()
)


positive_wtdrd1 = (
    df.loc[
        df["WTDRD1"] > 0,
        "WTDRD1"
    ]
)


actual_wtdrd1_min = float(
    positive_wtdrd1.min()
)

actual_wtdrd1_max = float(
    positive_wtdrd1.max()
)


print("\n--- WTDRD1 ---")

print(
    "Positive:",
    f"{wtdrd1_positive:,}"
)

print(
    "Zero:",
    f"{wtdrd1_zero:,}"
)

print(
    "Missing:",
    f"{wtdrd1_missing:,}"
)

print(
    "Exact XPT minimum:",
    repr(actual_wtdrd1_min)
)

print(
    "Exact XPT maximum:",
    repr(actual_wtdrd1_max)
)


if wtdrd1_positive != 8506:
    raise ValueError(
        "WTDRD1 positive count differs."
    )


if wtdrd1_zero != 1038:
    raise ValueError(
        "WTDRD1 zero count differs."
    )


if wtdrd1_missing != 0:
    raise ValueError(
        "WTDRD1 unexpected missing values."
    )


if actual_wtdrd1_min != 1974.178694:
    raise ValueError(
        "WTDRD1 exact XPT minimum differs."
    )


if actual_wtdrd1_max != 454277.717307:
    raise ValueError(
        "WTDRD1 exact XPT maximum differs."
    )


#
# CDC display validation only.
#
# Temporary scalar rounding; dataframe NOT altered.
#

if round(
    actual_wtdrd1_max,
    5
) != 454277.71731:

    raise ValueError(
        "WTDRD1 maximum does not match "
        "CDC displayed precision."
    )


print(
    "PASS: WTDRD1 exact XPT precision preserved."
)


# ============================================================
# 14. WTDR2D EXACT VALIDATION
# ============================================================

#
# Actual XPT and CDC:
#
#     positive min = 1663.514111
#     positive max = 579148.95623
#
#     positive = 7,027
#     zero     = 1,479
#     missing  = 1,038
#

wtdr2d_positive = int(
    df["WTDR2D"]
    .gt(0)
    .sum()
)

wtdr2d_zero = int(
    df["WTDR2D"]
    .eq(0)
    .sum()
)

wtdr2d_missing = int(
    df["WTDR2D"]
    .isna()
    .sum()
)


positive_wtdr2d = (
    df.loc[
        df["WTDR2D"] > 0,
        "WTDR2D"
    ]
)


actual_wtdr2d_min = float(
    positive_wtdr2d.min()
)

actual_wtdr2d_max = float(
    positive_wtdr2d.max()
)


print("\n--- WTDR2D ---")

print(
    "Positive:",
    f"{wtdr2d_positive:,}"
)

print(
    "Zero:",
    f"{wtdr2d_zero:,}"
)

print(
    "Missing:",
    f"{wtdr2d_missing:,}"
)

print(
    "Exact XPT minimum:",
    repr(actual_wtdr2d_min)
)

print(
    "Exact XPT maximum:",
    repr(actual_wtdr2d_max)
)


if wtdr2d_positive != 7027:
    raise ValueError(
        "WTDR2D positive count differs."
    )


if wtdr2d_zero != 1479:
    raise ValueError(
        "WTDR2D zero count differs."
    )


if wtdr2d_missing != 1038:
    raise ValueError(
        "WTDR2D missing count differs."
    )


if actual_wtdr2d_min != 1663.514111:
    raise ValueError(
        "WTDR2D exact XPT minimum differs."
    )


if actual_wtdr2d_max != 579148.95623:
    raise ValueError(
        "WTDR2D exact XPT maximum differs."
    )


print(
    "PASS: WTDR2D exact XPT range validated."
)


# ============================================================
# 15. CDC CHECK - DR2DRSTZ
# ============================================================

#
# Dietary recall status.
#

EXPECTED_DR2DRSTZ = {
    1: 6875,
    2: 44,
    4: 152,
    5: 2473
}


actual_dr2drstz = (
    df["DR2DRSTZ"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- DR2DRSTZ ---")

print(
    actual_dr2drstz
)


if actual_dr2drstz != EXPECTED_DR2DRSTZ:
    raise ValueError(
        "DR2DRSTZ frequencies differ."
    )


if int(df["DR2DRSTZ"].isna().sum()) != 0:
    raise ValueError(
        "Unexpected missing DR2DRSTZ."
    )


print(
    "PASS: DR2DRSTZ matches CDC."
)


# ============================================================
# 16. CDC CHECK - DRDINT
# ============================================================

#
# Number of intake days:
#
#     1 = Day 1 only
#     2 = Day 1 + Day 2
#

EXPECTED_DRDINT = {
    1: 1479,
    2: 7027
}


actual_drdint = (
    df["DRDINT"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_drdint != EXPECTED_DRDINT:
    raise ValueError(
        "DRDINT frequencies differ."
    )


if int(df["DRDINT"].isna().sum()) != 1038:
    raise ValueError(
        "DRDINT missing count differs."
    )


print(
    "PASS: DRDINT matches CDC."
)


# ============================================================
# 17. DRDINT / WTDR2D CONSISTENCY
# ============================================================

#
# DRDINT = 2 should exactly identify participants
# with a positive two-day dietary weight.
#

two_days = (
    df["DRDINT"]
    .eq(2)
    .fillna(False)
    .to_numpy(dtype=bool)
)


positive_two_day_weight = (
    df["WTDR2D"]
    .gt(0)
    .fillna(False)
    .to_numpy(dtype=bool)
)


two_day_mismatch = int(
    np.sum(
        two_days
        !=
        positive_two_day_weight
    )
)


print(
    "\nDRDINT / WTDR2D mismatches:",
    two_day_mismatch
)


if two_day_mismatch != 0:
    raise ValueError(
        "DRDINT=2 does not exactly match "
        "positive WTDR2D."
    )


print(
    "PASS: DRDINT and WTDR2D routing validated."
)


# ============================================================
# 18. CDC CHECK - DR2DBIH
# ============================================================

#
# Days between Day-2 intake and family interview.
#
# CDC range:
#
#     -28 to 141
#
# Attached XPT includes exactly 11 legitimate zeros.
#

dr2dbih_range = int(
    df["DR2DBIH"]
    .between(
        -28,
        141
    )
    .sum()
)


dr2dbih_zero = int(
    df["DR2DBIH"]
    .eq(0)
    .sum()
)


dr2dbih_missing = int(
    df["DR2DBIH"]
    .isna()
    .sum()
)


print("\n--- DR2DBIH ---")

print(
    "Range count:",
    dr2dbih_range
)

print(
    "Zero:",
    dr2dbih_zero
)

print(
    "Missing:",
    dr2dbih_missing
)


if dr2dbih_range != 6923:
    raise ValueError(
        "DR2DBIH range count differs."
    )


if dr2dbih_zero != 11:
    raise ValueError(
        "DR2DBIH zero count differs."
    )


if dr2dbih_missing != 2621:
    raise ValueError(
        "DR2DBIH missing count differs."
    )


print(
    "PASS: DR2DBIH validated."
)


# ============================================================
# 19. CDC CHECK - DR2TNUMF
# ============================================================

#
# Total number of foods/beverages reported.
#
# Range:
#
#     1-48
#
# n = 7,027
#

dr2tnumf_range = int(
    df["DR2TNUMF"]
    .between(
        1,
        48
    )
    .sum()
)


dr2tnumf_missing = int(
    df["DR2TNUMF"]
    .isna()
    .sum()
)


if dr2tnumf_range != 7027:
    raise ValueError(
        "DR2TNUMF range count differs."
    )


if dr2tnumf_missing != 2517:
    raise ValueError(
        "DR2TNUMF missing count differs."
    )


if int(df["DR2TNUMF"].eq(0).sum()) != 0:
    raise ValueError(
        "Unexpected DR2TNUMF zero."
    )


print(
    "PASS: DR2TNUMF 1-48 range validated."
)


# ============================================================
# 20. CDC CHECK - DR2_300
# ============================================================

#
# Compare amount eaten yesterday with usual amount.
#

EXPECTED_DR2_300 = {
    1: 397,
    2: 5759,
    3: 876,
    7: 1,
    9: 38
}


actual_dr2_300 = (
    df["DR2_300"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_dr2_300 != EXPECTED_DR2_300:
    raise ValueError(
        "DR2_300 frequencies differ."
    )


if int(df["DR2_300"].isna().sum()) != 2473:
    raise ValueError(
        "DR2_300 missing count differs."
    )


print(
    "PASS: DR2_300 frequencies validated."
)


# ============================================================
# 21. DEFINE 65 NUTRIENT / COMPONENT VARIABLES
# ============================================================

start_index = EXPECTED_COLUMNS.index(
    "DR2TKCAL"
)

end_index = EXPECTED_COLUMNS.index(
    "DR2TP226"
)


NUTRIENT_COLUMNS = (
    EXPECTED_COLUMNS[
        start_index:
        end_index + 1
    ]
)


if len(NUTRIENT_COLUMNS) != 65:
    raise ValueError(
        "Expected exactly 65 nutrient/component fields."
    )


print(
    "\nNutrient/component variables:",
    len(NUTRIENT_COLUMNS)
)


# ============================================================
# 22. NUTRIENT COMPLETENESS CHECK
# ============================================================

#
# Reliable Day-2 recalls:
#
#     DR2DRSTZ = 1
#     n = 6,875
#
# Every nutrient/component variable should have:
#
#     nonmissing = 6,875
#     missing    = 2,669
#

nutrient_completeness_errors = {}


for col in NUTRIENT_COLUMNS:

    nonmissing_count = int(
        df[col]
        .notna()
        .sum()
    )

    missing_count = int(
        df[col]
        .isna()
        .sum()
    )

    if not (
        nonmissing_count == 6875
        and
        missing_count == 2669
    ):

        nutrient_completeness_errors[col] = {
            "nonmissing": nonmissing_count,
            "missing": missing_count
        }


if nutrient_completeness_errors:
    raise ValueError(
        "Nutrient completeness pattern differs:\n"
        + str(
            nutrient_completeness_errors
        )
    )


print(
    "PASS: All 65 nutrient/component fields "
    "have 6,875 nonmissing and 2,669 missing records."
)


# ============================================================
# 23. SELECTED VALID ZERO COUNTS
# ============================================================

#
# These demonstrate that tiny values represent real zero,
# not missing values.
#

SELECTED_ZERO_COUNTS = {
    "DR2TKCAL": 5,
    "DR2TATOA": 5735,
    "DR2TB12A": 4566,
    "DR2TCAFF": 2175,
    "DR2TTHEO": 4035,
    "DR2TALCO": 6232
}


print("\n--- SELECTED ZERO COUNTS ---")


for col, expected_count in SELECTED_ZERO_COUNTS.items():

    actual_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    print(
        f"{col}:",
        f"{actual_count:,}"
    )

    if actual_count != expected_count:
        raise ValueError(
            f"{col}: zero count differs."
        )


print(
    "PASS: Selected nutrient zero counts validated."
)


# ============================================================
# 24. WATER VARIABLES
# ============================================================

#
# These are genuine Dietary decimal variables.
#
# DO NOT convert to Int64.
#
# CDC documents these as gram amounts and their valid
# ranges explicitly include zero.
#

WATER_CHECKS = {
    "DR2_320Z": {
        "low": 0,
        "high": 11520,
        "nonmissing": 7027,
        "missing": 2517,
        "zero": 1320
    },

    "DR2_330Z": {
        "low": 0,
        "high": 10560,
        "nonmissing": 7027,
        "missing": 2517,
        "zero": 3761
    },

    "DR2BWATZ": {
        "low": 0,
        "high": 11520,
        "nonmissing": 7027,
        "missing": 2517,
        "zero": 4084
    }
}


print("\n--- WATER VARIABLE CHECK ---")


for col, info in WATER_CHECKS.items():

    range_count = int(
        df[col]
        .between(
            info["low"],
            info["high"]
        )
        .sum()
    )

    missing_count = int(
        df[col]
        .isna()
        .sum()
    )

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    print(
        f"{col}: "
        f"range={range_count:,}, "
        f"missing={missing_count:,}, "
        f"zero={zero_count:,}"
    )

    if range_count != info["nonmissing"]:
        raise ValueError(
            f"{col}: valid range count differs."
        )

    if missing_count != info["missing"]:
        raise ValueError(
            f"{col}: missing count differs."
        )

    if zero_count != info["zero"]:
        raise ValueError(
            f"{col}: zero count differs."
        )


print(
    "PASS: Water variables validated."
)


# ============================================================
# 25. DR2TWS TAP WATER SOURCE
# ============================================================

EXPECTED_DR2TWS = {
    1: 4456,
    2: 503,
    3: 56,
    4: 1686,
    99: 370
}


actual_dr2tws = (
    df["DR2TWS"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_dr2tws != EXPECTED_DR2TWS:
    raise ValueError(
        "DR2TWS frequencies differ."
    )


if int(df["DR2TWS"].isna().sum()) != 2473:
    raise ValueError(
        "DR2TWS missing count differs."
    )


print(
    "PASS: DR2TWS frequencies validated."
)


# ============================================================
# 26. CONFIRM ALL 50 DECIMAL VARIABLES REMAIN FLOAT
# ============================================================

print("\n--- DECIMAL DTYPE CHECK ---")


decimal_dtype_failures = []


for col in DECIMAL_COLUMNS:

    if not pd.api.types.is_float_dtype(
        df[col].dtype
    ):

        decimal_dtype_failures.append(
            col
        )


if decimal_dtype_failures:
    raise ValueError(
        "Some genuine Dietary decimal variables "
        "are no longer floating point:\n"
        + str(decimal_dtype_failures)
    )


print(
    "PASS: All 50 genuine Dietary decimal fields "
    "remain floating-point."
)


# ============================================================
# 27. FINAL VALIDATED ZERO PATTERN
# ============================================================

#
# Raw pandas-decoded XPT contained no ordinary zeros.
#
# Therefore every zero after correction should correspond
# exactly to one validated tiny-value position.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:
        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    "Variables containing zeros:",
    len(final_zero_counts)
)

print(
    "Total zeros:",
    f"{sum(final_zero_counts.values()):,}"
)


if final_zero_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "Final zero pattern differs from "
        "validated DR2TOT_I pattern."
    )


if sum(final_zero_counts.values()) != 47411:
    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 47,411 validated restorations."
)


# ============================================================
# 28. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# The ONLY numerical changes permitted:
#
#     validated tiny -> 0
#
# All other values, including all Dietary decimals,
# must remain exactly numerically equal.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:
        pre_export_failures.append(
            col
        )


if pre_export_failures:
    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(pre_export_failures)
    )


print(
    "PASS: All 85 variables match "
    "the exact corrected XPT values."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# CRITICAL DIETARY EXPORT:
#
# NO:
#
#     df.round(...)
#
# NO:
#
#     float_format=
#
# NO:
#
#     conversion of true decimal fields to integers
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 30. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:
    raise ValueError(
        "CSV dimensions changed."
    )


if roundtrip_df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 31. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print(
    "\n--- EXACT CORRECTED XPT -> CSV FIDELITY ---"
)


csv_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:
        csv_fidelity_failures.append(
            col
        )


if csv_fidelity_failures:
    raise ValueError(
        "Corrected XPT -> CSV numeric differences:\n"
        + str(csv_fidelity_failures)
    )


print(
    "PASS: 0 numeric differences across "
    "all 85 variables."
)


# ============================================================
# 32. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING-VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:
    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 33. READ FINAL CSV AS RAW TEXT
# ============================================================

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 34. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(
            TINY_VALUE
        )
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:
    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 35. .0 CHECK - INTEGER/CODE FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT run this against all 85 columns.
#
# Genuine Dietary decimal columns may legitimately
# contain:
#
#     0.0
#     1.0
#     5.0
#
# Check only the 35 proven integer/code fields.
#

print("\n--- INTEGER FIELD .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:
        dot_zero_errors[col] = count


if dot_zero_errors:
    raise ValueError(
        "Unexpected '.0' formatting in "
        "whole-number/code fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 35 whole-number/code fields."
)


# ============================================================
# 36. LEGITIMATE DECIMAL .0 VALUES
# ============================================================

#
# INFORMATIONAL ONLY.
#
# Do not treat '.0' inside genuine Dietary decimal columns
# as an error.
#

print("\n--- LEGITIMATE DECIMAL .0 TOKENS ---")


decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col].ne("")
            &
            raw_csv_df[col].str.endswith(".0")
        )
        .sum()
    )

    decimal_dot_zero_counts[col] = count

    print(
        f"{col}:",
        f"{count:,}"
    )


print(
    "PASS: Genuine decimal '.0' values "
    "are preserved."
)


# ============================================================
# 37. EXACT DIETARY DECIMAL FIDELITY
# ============================================================

#
# Most important Dietary check.
#
# rtol = 0
# atol = 0
#
# No numeric difference is accepted.
#

print(
    "\n--- EXACT DIETARY DECIMAL FIDELITY ---"
)


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:
        decimal_failures.append(
            col
        )


if decimal_failures:
    raise ValueError(
        "Dietary decimal fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All 50 genuine Dietary decimal fields "
    "preserved exactly."
)


# ============================================================
# 38. SPECIFIC WTDRD1 CSV PRECISION CHECK
# ============================================================

#
# Preserve the actual XPT precision:
#
#     454277.717307
#
# Do NOT change it to the shorter CDC display:
#
#     454277.71731
#

csv_wtdrd1_max = float(
    roundtrip_df.loc[
        roundtrip_df["WTDRD1"] > 0,
        "WTDRD1"
    ]
    .max()
)


print("\n--- WTDRD1 CSV PRECISION ---")

print(
    "Final CSV numeric maximum:",
    repr(csv_wtdrd1_max)
)


if csv_wtdrd1_max != 454277.717307:
    raise ValueError(
        "WTDRD1 precision changed during CSV export."
    )


print(
    "PASS: WTDRD1 maximum remains exactly "
    "454277.717307."
)


# ============================================================
# 39. SPECIFIC WTDR2D CSV PRECISION CHECK
# ============================================================

csv_wtdr2d_min = float(
    roundtrip_df.loc[
        roundtrip_df["WTDR2D"] > 0,
        "WTDR2D"
    ]
    .min()
)


csv_wtdr2d_max = float(
    roundtrip_df.loc[
        roundtrip_df["WTDR2D"] > 0,
        "WTDR2D"
    ]
    .max()
)


if csv_wtdr2d_min != 1663.514111:
    raise ValueError(
        "WTDR2D minimum changed during export."
    )


if csv_wtdr2d_max != 579148.95623:
    raise ValueError(
        "WTDR2D maximum changed during export."
    )


print(
    "PASS: WTDR2D exact precision preserved."
)


# ============================================================
# 40. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0)
        .sum()
    )

    if zero_count > 0:
        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    "Variables containing zeros:",
    len(csv_zero_counts)
)

print(
    "Total zero values:",
    f"{sum(csv_zero_counts.values()):,}"
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:
    raise ValueError(
        "Final CSV zero pattern differs from "
        "validated corrected source."
    )


print(
    "PASS: Final CSV contains exactly "
    "47,411 validated zero values."
)


# ============================================================
# 41. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]

    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:
    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 42. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 83732:
    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 93702:
    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):
    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no '.0'."
)


# ============================================================
# 43. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 100
)

print(
    "DR2TOT_I CONVERSION COMPLETE"
)

print(
    "=" * 100
)

print(
    "Cycle: 2015-2016"
)

print(
    "Component: Dietary - "
    "Total Nutrient Intakes, Second Day"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)

print(
    "Variables containing tiny artifacts:",
    tiny_variable_count
)

print(
    "Validated zero restorations:",
    f"{correction_total:,}"
)

print(
    "Genuine Dietary decimal fields:",
    len(DECIMAL_COLUMNS)
)

print(
    "Whole-number/code fields:",
    len(INTEGER_COLUMNS)
)

print(
    "WTDRD1 positive:",
    f"{wtdrd1_positive:,}"
)

print(
    "WTDRD1 zero:",
    f"{wtdrd1_zero:,}"
)

print(
    "WTDRD1 exact XPT minimum:",
    repr(actual_wtdrd1_min)
)

print(
    "WTDRD1 exact XPT maximum:",
    repr(actual_wtdrd1_max)
)

print(
    "WTDR2D positive:",
    f"{wtdr2d_positive:,}"
)

print(
    "WTDR2D zero:",
    f"{wtdr2d_zero:,}"
)

print(
    "WTDR2D missing:",
    f"{wtdr2d_missing:,}"
)

print(
    "WTDR2D exact minimum:",
    repr(actual_wtdr2d_min)
)

print(
    "WTDR2D exact maximum:",
    repr(actual_wtdr2d_max)
)

print(
    "Reliable Day-2 recalls:",
    int(
        df["DR2DRSTZ"]
        .eq(1)
        .sum()
    )
)

print(
    "Nutrient/component variables:",
    len(NUTRIENT_COLUMNS)
)

print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)

print(
    "Missing-position differences:",
    "0 expected"
)

print(
    "Dietary decimal differences:",
    "0 expected"
)

print(
    "Tiny values remaining:",
    "0 expected"
)

print(
    "Unnecessary '.0' in integer fields:",
    "0 expected"
)

print(
    "Decimal '.0' values:",
    "ALLOWED / PRESERVED"
)

print(
    "Scientific notation:",
    "0 expected"
)

print(
    "\nFINAL RULE:"
)

print(
    "All genuine Dietary decimal values remain "
    "exactly AS IS."
)

print(
    "No Dietary decimal value was rounded."
)

NHANES 2015-2016 DR2TOT_I XPT -> CSV
Rows: 9,544
Columns: 85
PASS: Structure = 9,544 rows x 85 columns.

--- FIELD TYPE PLAN ---
Genuine Dietary decimal fields: 50
Whole-number/code fields: 35

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT artifacts: 47,411
Variables containing tiny artifacts: 70

Tiny values by variable:
DR2TALCO    6232
DR2TATOA    5735
DR2TB12A    4566
DR2BWATZ    4084
DR2TTHEO    4035
DR2_330Z    3761
DR2TP184    3348
DR2TCAFF    2175
DR2TLYCO    1857
WTDR2D      1479
DR2_320Z    1320
DR2TP226    1110
DR2TM221    1087
WTDRD1      1038
DR2TACAR     950
DR2TP205     947
DR2TCRYP     704
DR2TP225     681
DR2TS060     328
DR2TS040     318
DR2TFA       239
DR2TVD       184
DR2TBCAR     124
DR2TS080     117
DR2TCHOL     104
DR2TLZ        90
DR2TFIBE      78
DR2TFF        69
DR2TRET       62
DR2TP204      61
DR2TVC        59
DR2TM201      42
DR2TS100      39
DR2TS120      37
DR2TVB12      33
DR2TVARA      30
DR2TM161      22
DR2TVK